# econ-pdf-extract: claims and equations with page numbers

Python API demo, no CLI required:

- runs on the synthetic `data/raw/example_paper.pdf` and Shapiro, Sudhof and Wilson, "Measuring News Sentiment", FRBSF Working Paper 2017-01, downloaded on first run and hash-checked
- paths resolve from the repo root (searched upward for `pyproject.toml`), not from the kernel CWD

Adapted from [https://github.com/corybaird/econ-pdf-extract/blob/main/notebooks/quickstart.ipynb](https://github.com/corybaird/econ-pdf-extract/blob/main/notebooks/quickstart.ipynb).

In [1]:
import hashlib
import os
import urllib.request
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("repo root:", REPO_ROOT.name)

INPUTS = yaml.safe_load((REPO_ROOT / "references/configs/notebook_inputs.yaml").read_text())


def fetch(key):
    spec = INPUTS[key]
    path = REPO_ROOT / spec["path"]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(spec["url"], headers={"User-Agent": "Mozilla/5.0"})
        path.write_bytes(urllib.request.urlopen(request, timeout=60).read())
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != spec["sha256"]:
        raise ValueError(f"{path.name}: hash mismatch {digest}")
    return path

repo root: workshop-notebooks


## 1. Run the pipeline on the synthetic example paper

`ExtractionPipeline.run()` returns one `ExtractionResult` with claims, math objects, symbols, links, and statistics.

In [2]:
from econ_pdf_extract import ExtractionPipeline

result = ExtractionPipeline(REPO_ROOT / "data/raw/example_paper.pdf").run()
print(
    f"pages={result.page_count} claims={len(result.claims)} math={len(result.math_objects)} "
    f"symbols={len(result.symbols)} links={len(result.links)} statistics={len(result.statistics)}"
)

pages=2 claims=5 math=0 symbols=0 links=0 statistics=8


## 2. Claims with page / section provenance

Every claim keeps its PDF provenance (`page`, `section` in the table; `bbox`, `printed_page`, `source_channel` are on the object itself).

In [3]:
def short(text, n=72):
    return text if len(text) <= n else text[: n - 3] + "..."

for claim in result.claims:
    types = ", ".join(t.value for t in claim.claim_types)
    print(f"{claim.claim_id} p{claim.page} [{types}] ({claim.attribution.value}) {claim.section}")
    print("   ", short(claim.quote))

CLM-001 p1 [mechanism, magnitude] (own) 1 Introduction
    We find that monetary policy surprises reduce industrial production b...
CLM-002 p1 [significance, magnitude] (own) 1 Introduction
    The estimate on the policy surprise is -0.42 and is significant at th...
CLM-003 p1 [robustness] (own) 1 Introduction
    Our results show that this estimate is robust to alternative identifi...
CLM-004 p2 [significance, magnitude] (unclear) 2 Results and Diagnostics
    The regression R-squared is 0.21, the t-statistic is 3.45, and N = 21...
CLM-005 p2 [magnitude] (unclear) 2 Results and Diagnostics
    The 95% CI is [-1.8, -0.9] percent wide, and the placebo regression y...


## 3. Math objects, symbols, and links on a public FRBSF paper

A longer paper with real display equations: tags, kinds, and symbol definitions.

In [4]:
from econ_pdf_extract import ExtractionPipeline

shapiro = ExtractionPipeline(fetch("shapiro_wp2017_01")).run()
print(
    f"claims={len(shapiro.claims)} math={len(shapiro.math_objects)} "
    f"symbols={len(shapiro.symbols)} links={len(shapiro.links)}"
)

claims=22 math=9 symbols=1 links=0


In [5]:
for eq in shapiro.math_objects[:6]:
    tag = eq.tag or "-"
    print(f"{eq.math_id} p{eq.page} {eq.kind.value} {tag}")
    print("   ", short(eq.raw_text.replace(chr(10), " ")))

print(f"symbols: {len(shapiro.symbols)}, links: {len(shapiro.links)}")

eqs = shapiro.math_objects
eq_demo = eqs[0]
print("demo equation for step 4:", eq_demo.math_id, eq_demo.tag)

EQ-001 p17 display_equation (1)
    p(w, c)  PMI(w, c) = log p(w)p(c) where, p(w) is word w’s share of to...
EQ-002 p17 display_equation (2)
    S(w) = PMI(w, positive) −PMI(w, negative),
EQ-003 p21 display_equation (3)
    sa = ft(a) + fp(a),j(a) + εa.
EQ-004 p23 display_equation (4)
    3 3 Prelimt = a +XγτFinalt−τ + βdNStd +XφτNSt−τ + εt, τ=1 τ=1
EQ-005 p25 display_equation (5)
    4 4 yi,t+h = ahi + βihNSt +Xαi,τh NSt−l + Ahi XYt−τ + εi,t+h. τ=1 τ=0
EQ-006 p26 estimation_problem (6)
    n p !2 p βˆ = arg minX yi,t+12 −Xxi,jbj + λX|bj| i=1 j=1 j=1
symbols: 1, links: 0
demo equation for step 4: EQ-001 (1)


## 4. A linked equation–claim pair

`CrossReferenceLinker` adds bidirectional edges between claims and equations (`via`: `explicit_ref`, `symbol`, or `named_metric`). No sentence in the public samples cites an equation tag, so the claim below is a constructed demo input; the equation is a real pipeline object from step 3.

In [6]:
from econ_pdf_extract import Claim, ClaimType, CrossReferenceLinker, SectionRole
from econ_pdf_extract.core.models import Attribution

claim = Claim(
    claim_id="CLM-DEMO",
    paper_id=shapiro.paper.paper_id,
    claim_types=[ClaimType.MECHANISM],
    attribution=Attribution.OWN,
    page=eq_demo.page,
    page_end=eq_demo.page,
    section=eq_demo.section,
    section_role=SectionRole.RESULTS,
    bbox=eq_demo.bbox,
    quote="The sentiment lexicon follows from the pointwise mutual information defined in Equation (1).",
)
links = CrossReferenceLinker().link([claim], eqs, shapiro.symbols)
edge = links[0]

print(f"edge: {edge.claim_id} <-> {edge.math_id} (via {edge.via}, token {edge.matched_token!r})")
print("claim:", short(claim.quote))
print(f"equation [{edge.math_id}] p{eq_demo.page} {eq_demo.tag or '-'}:", short(eq_demo.raw_text.replace(chr(10), " "), 60))
print("claim.linked_equation_ids:", claim.linked_equation_ids, "| eq.linked_claim_ids:", eq_demo.linked_claim_ids)

edge: CLM-DEMO <-> EQ-001 (via explicit_ref, token '(1)')
claim: The sentiment lexicon follows from the pointwise mutual information d...
equation [EQ-001] p17 (1): p(w, c)  PMI(w, c) = log p(w)p(c) where, p(w) is word w’s...
claim.linked_equation_ids: ['EQ-001'] | eq.linked_claim_ids: ['CLM-DEMO']


## 5. JSON export

`JsonReportWriter` serializes the full result with provenance. Truncated preview here; run `--output-dir` for real files.

In [7]:
import json

from econ_pdf_extract import JsonReportWriter

payload = json.loads(JsonReportWriter().to_json(shapiro))
out = {
    "paper": payload["paper"]["paper_id"],
    "counts": payload["diagnostics"],
    "first_claim": payload["claims"][0],
}
print(json.dumps(out, indent=2)[:900])

{
  "paper": "shapiro_2017-01_sentiment",
  "counts": {
    "total_sentences": 553,
    "total_claims": 22,
    "canonical_claims": 22,
    "duplicate_claims": 0,
    "math_objects": 9,
    "needs_ocr_displays": 7,
    "ocr_calls": 0,
    "symbols": 1,
    "links": 0,
    "statistics": 37,
    "causal_triples": 0,
    "duplicate_front_matter_pages": [],
    "warnings": []
  },
  "first_claim": {
    "claim_id": "CLM-001",
    "paper_id": "shapiro_2017-01_sentiment",
    "claim_types": [
      "proposal"
    ],
    "attribution": "own",
    "page": 2,
    "page_end": 2,
    "section": "Abstract",
    "section_role": "abstract",
    "bbox": [
      101.2649917602539,
      304.103271484375,
      510.78167724609375,
      348.641357421875
    ],
    "quote": "This paper demonstrates state-of-the-art text sentiment analysis tools while developing a new time-series measure of economic sentim
